# Day 9 v3 — Silver: All Report Types (Production — Job Parameters Only)

**Production notebook. Must be run via Databricks Job or ADF pipeline.**
**Interactive run will fail at Cell 2 — by design.**

---

## What this notebook does

Reads all three monthly report JSON files from Bronze, flattens their nested schemas,
applies data quality checks, and writes to three separate Silver Delta tables.

| Report | Bronze file | Silver table | Grain |
|---|---|---|---|
| kpi_report | `kpi_report_YYYYMM.json` | `silver-volume/reports/kpi_report` | 1 row/month |
| sla_report | `sla_report_YYYYMM.json` | `silver-volume/reports/sla_report` | 1 row/month |
| state_breakdown | `state_breakdown_YYYYMM.json` | `silver-volume/reports/state_breakdown` | 1 row/state/month |

---

## Parameters (injected by Databricks Job)

| Parameter | Required | Example | Notes |
|---|---|---|---|
| `load_type` | Yes | `incremental` | `full` or `incremental` |
| `ingestion_year` | Yes | `2026` | YYYY |
| `ingestion_month` | Yes | `06` | MM (zero-padded) |

---

## MERGE keys

| Silver table | MERGE key |
|---|---|
| kpi_report | `report_period` |
| sla_report | `report_period` |
| state_breakdown | `report_period` + `state_code` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Parameters
# Only load_type is required from ADF / Databricks Job.
# Year/month default to the PREVIOUS UTC month — monthly report files are generated at
# month-end for the prior month, so when this job runs it processes last month's reports.
#
#   load_type       : "full" | "incremental"  (pass from ADF, default: incremental)
#   ingestion_year  : YYYY                    (default: previous UTC month's year)
#   ingestion_month : MM                      (default: previous UTC month, zero-padded)

from datetime import timedelta
_now        = datetime.now(timezone.utc)
# Subtract enough days to land in the previous month (any day of current month - 32 days
# is guaranteed to be in the prior month), then take year/month from that date.
_prev_month = (_now.replace(day=1) - timedelta(days=1))  # last day of previous month

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE       = _get_param("load_type",       "incremental")
INGESTION_YEAR  = _get_param("ingestion_year",  str(_prev_month.year))
INGESTION_MONTH = _get_param("ingestion_month", f"{_prev_month.month:02d}")

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print("Parameters resolved.")

In [ ]:
# Cell 3 — Constants
BRONZE_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/reports"
SILVER_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports"

YYYYMM         = f"{INGESTION_YEAR}{INGESTION_MONTH}"
PARTITION_PATH = f"{BRONZE_REPORTS}/{INGESTION_YEAR}/{INGESTION_MONTH}"

PIPELINE_NAME  = "job_silver_blob_reports_v3"
RUN_TS         = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze partition : {PARTITION_PATH}")
print(f"Silver base      : {SILVER_REPORTS}")
print(f"RUN_TS           : {RUN_TS}")

In [ ]:
# Cell 4 — Helper functions

def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def write_quarantine(df, reject_reason, quarantine_path):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reject_reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .save(quarantine_path)
    )

print("Helper functions defined.")

In [ ]:
# Cell 5 — Flatten functions per report type

def flatten_kpi(raw_df):
    return raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.col("summary.total_sessions").cast("long").alias("total_sessions"),
        F.col("summary.total_energy_kwh").cast("decimal(14,2)").alias("total_energy_kwh"),
        F.col("summary.total_revenue_aud").cast("decimal(14,2)").alias("total_revenue_aud"),
        F.col("summary.avg_session_duration_min").cast("decimal(8,2)").alias("avg_session_duration_min"),
        F.col("summary.avg_energy_per_session_kwh").cast("decimal(8,2)").alias("avg_energy_per_session_kwh"),
        F.lit(int(INGESTION_YEAR)).alias("report_year"),
        F.lit(int(INGESTION_MONTH)).alias("report_month"),
        F.lit(RUN_TS).cast("timestamp").alias("silver_ingested_at"),
        F.lit(LOAD_TYPE).alias("silver_load_type"),
        F.lit(PIPELINE_NAME).alias("silver_pipeline"),
    )


def flatten_sla(raw_df):
    return raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.col("sla_metrics.uptime_pct").cast("decimal(6,3)").alias("uptime_pct"),
        F.col("sla_metrics.avg_response_time_sec").cast("decimal(8,2)").alias("avg_response_time_sec"),
        F.col("sla_metrics.incidents").cast("integer").alias("incidents"),
        F.col("sla_metrics.mttr_hours").cast("decimal(8,2)").alias("mttr_hours"),
        F.col("sla_metrics.chargers_meeting_sla").cast("integer").alias("chargers_meeting_sla"),
        F.col("sla_metrics.total_chargers").cast("integer").alias("total_chargers"),
        F.lit(int(INGESTION_YEAR)).alias("report_year"),
        F.lit(int(INGESTION_MONTH)).alias("report_month"),
        F.lit(RUN_TS).cast("timestamp").alias("silver_ingested_at"),
        F.lit(LOAD_TYPE).alias("silver_load_type"),
        F.lit(PIPELINE_NAME).alias("silver_pipeline"),
    )


def flatten_state_breakdown(raw_df):
    exploded = raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.explode(F.col("states")).alias("s")
    )
    return exploded.select(
        "report_period",
        "generated_at",
        F.col("s.state_code").cast("string").alias("state_code"),
        F.col("s.sessions").cast("long").alias("sessions"),
        F.col("s.energy_kwh").cast("decimal(14,2)").alias("energy_kwh"),
        F.col("s.revenue_aud").cast("decimal(14,2)").alias("revenue_aud"),
        F.lit(int(INGESTION_YEAR)).alias("report_year"),
        F.lit(int(INGESTION_MONTH)).alias("report_month"),
        F.lit(RUN_TS).cast("timestamp").alias("silver_ingested_at"),
        F.lit(LOAD_TYPE).alias("silver_load_type"),
        F.lit(PIPELINE_NAME).alias("silver_pipeline"),
    )


print("Flatten functions defined.")

In [ ]:
# Cell 6 — Report config list with MERGE logic

REPORTS = [
    {
        "report_type"   : "kpi_report",
        "file_name"     : f"kpi_report_{YYYYMM}.json",
        "flatten_fn"    : flatten_kpi,
        "silver_path"   : f"{SILVER_REPORTS}/kpi_report",
        "quarantine"    : f"{SILVER_REPORTS}/../quarantine/reports/kpi_report",
        "merge_key"     : "target.report_period = source.report_period",
        "null_check_col": "report_period",
    },
    {
        "report_type"   : "sla_report",
        "file_name"     : f"sla_report_{YYYYMM}.json",
        "flatten_fn"    : flatten_sla,
        "silver_path"   : f"{SILVER_REPORTS}/sla_report",
        "quarantine"    : f"{SILVER_REPORTS}/../quarantine/reports/sla_report",
        "merge_key"     : "target.report_period = source.report_period",
        "null_check_col": "report_period",
    },
    {
        "report_type"   : "state_breakdown",
        "file_name"     : f"state_breakdown_{YYYYMM}.json",
        "flatten_fn"    : flatten_state_breakdown,
        "silver_path"   : f"{SILVER_REPORTS}/state_breakdown",
        "quarantine"    : f"{SILVER_REPORTS}/../quarantine/reports/state_breakdown",
        # Composite merge key: one row per state per month
        "merge_key"     : "target.report_period = source.report_period AND target.state_code = source.state_code",
        "null_check_col": "state_code",
    },
]

print(f"Reports to process: {len(REPORTS)}")
for r in REPORTS:
    print(f"  {r['report_type']:<20} merge_key={r['merge_key']}")

In [ ]:
# Cell 7 — Verify all source files exist before starting

missing = []
for report in REPORTS:
    file_path = f"{PARTITION_PATH}/{report['file_name']}"
    try:
        dbutils.fs.ls(file_path)
        print(f"  [FOUND]  {file_path}")
    except Exception:
        print(f"  [MISSING] {file_path}")
        missing.append(file_path)

if missing:
    raise Exception(
        f"{len(missing)} report file(s) missing in Bronze.\n"
        f"Run day_6/05_bronze_blob_reports_json.ipynb first for "
        f"{INGESTION_YEAR}/{INGESTION_MONTH}."
    )

print("\nAll source files confirmed.")

In [ ]:
# Cell 8 — Main loop: read → flatten → data quality → write Silver

results = []

for report in REPORTS:
    rtype      = report["report_type"]
    file_path  = f"{PARTITION_PATH}/{report['file_name']}"
    silver_p   = report["silver_path"]
    quarantine = report["quarantine"]
    merge_key  = report["merge_key"]
    null_col   = report["null_check_col"]

    print(f"\nProcessing: {rtype}")
    print(f"  Source: {file_path}")

    try:
        # Step 1: Read Bronze JSON
        raw_df = (
            spark.read
            .option("multiLine", True)
            .json(file_path)
        )
        bronze_rows = raw_df.count()
        print(f"  Bronze rows : {bronze_rows}")

        # Step 2: Flatten with report-specific function
        flat_df = report["flatten_fn"](raw_df)

        # Step 3: Data quality — quarantine null key rows
        null_df  = flat_df.filter(F.col(null_col).isNull() | (F.col(null_col) == ""))
        write_quarantine(null_df, f"null_{null_col}", quarantine)
        clean_df = flat_df.filter(F.col(null_col).isNotNull() & (F.col(null_col) != ""))

        null_dropped = null_df.count()
        silver_rows  = clean_df.count()
        if null_dropped > 0:
            print(f"  Quarantined (null_{null_col}) : {null_dropped}")

        # Step 4: Write to Silver (full overwrite or MERGE)
        if LOAD_TYPE == "full" or not folder_exists_dbfs(silver_p):
            (
                clean_df.write.format("delta")
                .mode("overwrite").option("overwriteSchema", "true")
                .save(silver_p)
            )
            print(f"  Full overwrite : {silver_rows} rows")
        else:
            delta_table = DeltaTable.forPath(spark, silver_p)
            (
                delta_table.alias("target")
                .merge(clean_df.alias("source"), merge_key)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            print(f"  MERGE upserted : {silver_rows} rows")

        results.append({"report": rtype, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as ex:
        print(f"  FAILED: {ex}")
        results.append({"report": rtype, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(ex)})

print("\nAll reports processed.")

In [ ]:
# Cell 9 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 65)
print("SILVER BLOB REPORTS v3 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp   : {RUN_TS}")
print(f"  load_type       : {LOAD_TYPE}")
print(f"  partition       : {INGESTION_YEAR}/{INGESTION_MONTH}")
print(f"  succeeded       : {len(succeeded)}")
print(f"  failed          : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['report']:<20} bronze={r['bronze_rows']:>4}  silver={r['silver_rows']:>4}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} report(s) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} reports written to Silver successfully.")